# Current BOAMP EDA

**Current dataset notebook.** This notebook has been refreshed for the enriched current BOAMP dataset under `data/processed/boamp_current/`.

Explores the enriched current BOAMP dataset. DECP and older 2015-2024 outputs are historical unless explicitly cited.

Interpretation guardrail: `event = 1` is a **proxy recurrence**, meaning an identifiable reappearance of a similar procurement need under the selected rule. It is not a legally verified renewal.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd()
PROJECT = cwd
for candidate in [cwd, *cwd.parents]:
    if (candidate / "data").exists() and (candidate / "reports").exists():
        PROJECT = candidate
        break

DATA = PROJECT / "data" / "processed" / "boamp_current"
RAW = PROJECT / "data" / "raw" / "boamp_current"
TABLES = PROJECT / "reports" / "tables"
FIGURES = PROJECT / "reports" / "figures"

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

print(f"Project root: {PROJECT}")
print(f"Current data directory: {DATA}")


## Current Dataset EDA


In [ ]:
download = pd.read_csv(TABLES / "data" / "boamp_current_download_summary.csv")
enrichment = pd.read_csv(TABLES / "data" / "buyer_enrichment_summary.csv")
population_summary = pd.read_csv(TABLES / "data" / "analytical_population_summary.csv")
selected = pd.read_csv(TABLES / "linkage" / "final_selected_event_definition_current.csv")
survival = pd.read_csv(TABLES / "survival" / "survival_summary_current.csv")
risk = pd.read_csv(TABLES / "survival" / "operational_risk_scores_current.csv", dtype={"SIREN": str, "SIRET": str})

summary = {
    "study_period": download["actual_date_range"].dropna().iloc[-1],
    "retained_notices": int(download["number_of_retained_notices"].dropna().iloc[-1]),
    "appel_offre": int(enrichment.loc[enrichment["metric"].eq("APPEL_OFFRE"), "value"].iloc[0]),
    "eligible_contracts": int(selected["eligible_contracts"].iloc[0]),
    "selected_method": f"{selected['selected_method'].iloc[0]} {selected['selected_variant'].iloc[0]}",
    "proxy_events": int(selected["event_count"].iloc[0]),
    "event_rate": float(selected["event_rate"].iloc[0]),
    "censoring_date": selected["censoring_date"].iloc[0],
    "survival_24m": float(survival["survival_24m"].iloc[0]),
    "mean_p12": float(risk["p_renewal_12m"].mean()),
    "mean_p24": float(risk["p_renewal_24m"].mean()),
}
pd.DataFrame([summary])


In [ ]:
clean = pd.read_csv(DATA / "boamp_full_clean_enriched.csv", parse_dates=["dateparution"], low_memory=False)
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
clean.groupby(clean["dateparution"].dt.year).size().plot.bar(ax=axes[0, 0], color="#4C78A8", title="Current notices by year")
clean["nature"].value_counts().plot.bar(ax=axes[0, 1], color="#72B7B2", title="Notice nature")
clean["buyer_key_type"].value_counts().plot.bar(ax=axes[1, 0], color="#F58518", title="Buyer key type")
clean["category_label"].fillna("Unknown").value_counts().head(10).sort_values().plot.barh(ax=axes[1, 1], color="#54A24B", title="Top segments")
plt.tight_layout()
plt.show()


In [ ]:
display(clean.groupby("year", dropna=False).agg(
    notices=("idweb", "count"),
    appel_offre=("nature", lambda s: (s == "APPEL_OFFRE").sum()),
    attribution=("nature", lambda s: (s == "ATTRIBUTION").sum()),
    siren_or_siret=("buyer_key_type", lambda s: s.ne("NAME").sum()),
    generic_cpv=("cpv_is_generic", "sum"),
).reset_index().tail(12))
